# 🚀 CIFAR-100 Training on Colab
自動掛載 Google Drive、clone repo、偵測 checkpoint 並恢復訓練。

## Step 1｜掛載 Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Checkpoint 存放路徑（Drive 上）
CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'✅ Checkpoint 目錄：{CHECKPOINT_DIR}')

ModuleNotFoundError: No module named 'google'

In [1]:
!git config --global user.name "rvgoing"
!git config --global user.email "cmkdkimo@gmail.com"

## Step 2｜Clone / Pull 最新程式碼

In [ ]:
GITHUB_REPO = 'https://github.com/rvgoing/CFIR_100_ReFresh.git'  # ← 改這裡
REPO_NAME   = 'CFIR_100_ReFresh'                           # ← 改這裡（資料夾名稱）

if os.path.exists(f'/content/{REPO_NAME}'):
    print('📦 Repo 已存在，執行 git pull ...')
    %cd /content/{REPO_NAME}
    !git pull
else:
    print('📦 Clone repo ...')
    !git clone {GITHUB_REPO}
    %cd /content/{REPO_NAME}

print('✅ 程式碼已是最新版本')

## Step 3｜安裝依賴

In [ ]:
!pip install -q -r requirements.txt
print('✅ 套件安裝完成')

## Step 4｜確認 GPU

In [ ]:
import torch
print(f'PyTorch 版本：{torch.__version__}')
print(f'CUDA 可用：{torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU：{torch.cuda.get_device_name(0)}')
    print(f'VRAM：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
!nvidia-smi

## Step 5｜自動偵測 Checkpoint 並開始訓練

In [ ]:
import os

# 偵測 Drive 上是否已有 checkpoint
BEST_CKPT = os.path.join(CHECKPOINT_DIR, 'model_best.pth.tar')
LAST_CKPT = os.path.join(CHECKPOINT_DIR, 'checkpoint.pth.tar')

# 優先 resume 最後一個 checkpoint（保留訓練進度），其次用 best
if os.path.exists(LAST_CKPT):
    resume_path = LAST_CKPT
    print(f'🔄 發現 checkpoint，將從上次進度恢復：{resume_path}')
elif os.path.exists(BEST_CKPT):
    resume_path = BEST_CKPT
    print(f'🔄 發現 best checkpoint，將從此恢復：{resume_path}')
else:
    resume_path = ''
    print('🆕 未發現 checkpoint，從頭開始訓練')

# 組合訓練指令
resume_arg = f'--resume {resume_path}' if resume_path else ''

cmd = f"""
python train.py \\
    --epochs 20 \\
    --batch-size 128 \\
    --lr 0.1 \\
    --save-dir {CHECKPOINT_DIR} \\
    {resume_arg}
""".strip()

print(f'\n🚀 執行指令：\n{cmd}\n')
!{cmd}

## Step 6｜確認 Checkpoint 已存到 Drive

In [ ]:
import os

print('📂 Drive checkpoint 目錄內容：')
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    size = os.path.getsize(os.path.join(CHECKPOINT_DIR, f)) / 1024**2
    print(f'  {f}  ({size:.1f} MB)')